# Đánh Giá Judge

Notebook này chạy judge cho một retrieval experiment cố định, lưu verdict/explanation, rồi tính metric cho nhãn verdict. Evidence dùng toàn bộ text URLs và chỉ top 1 image URL để tiết kiệm request.

## 1. Import Và Cấu Hình

In [1]:
# %pip install -q openai python-dotenv pandas tqdm pydantic

In [2]:
from __future__ import annotations

import ast
import asyncio
import json
import math
import os
import re
from pathlib import Path
from typing import Any, Literal

import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, ConfigDict
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 180)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "judge":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
CURRENT_MODEL_ID = "deepseek/deepseek-v4-flash"
MODEL_ALIAS = "deepseek-v4-flash"

client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

if not OPENROUTER_API_KEY:
    print("Chưa thấy OPENROUTER_API_KEY trong .env. Cell gọi API sẽ lỗi nếu chưa cấu hình key.")
else:
    print("Đã nạp OPENROUTER_API_KEY.")
print(f"Model judge: {CURRENT_MODEL_ID}")

Đã nạp OPENROUTER_API_KEY.
Model judge: deepseek/deepseek-v4-flash


## 2. Load CSV

In [3]:
EXPERIMENT_ID = "gemini-2.5-flash__semantic__clip_finetuned__reranker_1"
EXPERIMENT_DIR = PROJECT_ROOT / "database" / "retrieval_eval_outputs" / "by_experiment" / EXPERIMENT_ID

FILE_TOP3_URLS = EXPERIMENT_DIR / "top3_urls.csv"
FILE_REFINED = PROJECT_ROOT / "refined" / "refined_outputs_openrouter" / "refined_gemini-2.5-flash.csv"
FILE_CORPUS = PROJECT_ROOT / "chunking_scripts" / "final_corpus.csv"
FILE_GOLD = PROJECT_ROOT / "FinalDataset" / "claims_merged.csv"

OUTPUT_DIR = PROJECT_ROOT / "judge" / "judge_outputs_openrouter_deepseek_v4_flash"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = False
SMOKE_ROWS = 5
OUTPUT_SUFFIX = "smoke" if SMOKE_TEST else "full"

OUTPUT_FILE = OUTPUT_DIR / f"factchecking_final_results_{MODEL_ALIAS}__{EXPERIMENT_ID}__{OUTPUT_SUFFIX}.csv"
METRICS_FILE = OUTPUT_DIR / f"factchecking_metrics_{MODEL_ALIAS}__{EXPERIMENT_ID}__{OUTPUT_SUFFIX}.csv"
CONFUSION_FILE = OUTPUT_DIR / f"factchecking_confusion_{MODEL_ALIAS}__{EXPERIMENT_ID}__{OUTPUT_SUFFIX}.csv"
ERRORS_FILE = OUTPUT_DIR / f"factchecking_errors_{MODEL_ALIAS}__{EXPERIMENT_ID}__{OUTPUT_SUFFIX}.csv"

COL_CLAIM_ID = "id"
COL_TOP3_TEXT = "top3_text_urls"
COL_TOP3_IMAGE = "top3_image_urls"
COL_REFINED_JSON = "raw_output"
COL_CORPUS_URL = "url"
COL_CORPUS_TEXT = "content"

print(f"Top3 URLs: {FILE_TOP3_URLS.exists()} - {FILE_TOP3_URLS}")
print(f"Refined: {FILE_REFINED.exists()} - {FILE_REFINED}")
print(f"Corpus: {FILE_CORPUS.exists()} - {FILE_CORPUS}")
print(f"Gold labels: {FILE_GOLD.exists()} - {FILE_GOLD}")
print(f"Smoke test: {SMOKE_TEST}, smoke rows: {SMOKE_ROWS}")
print(f"Output: {OUTPUT_FILE}")

Top3 URLs: True - d:\FactCheckPipeline\database\retrieval_eval_outputs\by_experiment\gemini-2.5-flash__semantic__clip_finetuned__reranker_1\top3_urls.csv
Refined: True - d:\FactCheckPipeline\refined\refined_outputs_openrouter\refined_gemini-2.5-flash.csv
Corpus: True - d:\FactCheckPipeline\chunking_scripts\final_corpus.csv
Gold labels: True - d:\FactCheckPipeline\FinalDataset\claims_merged.csv
Smoke test: False, smoke rows: 5
Output: d:\FactCheckPipeline\judge\judge_outputs_openrouter_deepseek_v4_flash\factchecking_final_results_deepseek-v4-flash__gemini-2.5-flash__semantic__clip_finetuned__reranker_1__full.csv


In [4]:
def safe_json_loads(value, fallback):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return fallback
    if isinstance(value, (dict, list)):
        return value
    text = str(value).strip()
    if not text:
        return fallback
    try:
        return json.loads(text)
    except Exception:
        try:
            return ast.literal_eval(text)
        except Exception:
            return fallback


def dedupe_keep_order(values):
    seen = set()
    out = []
    for value in values:
        if value is None or (isinstance(value, float) and pd.isna(value)):
            continue
        value = str(value).strip()
        if not value or value in seen:
            continue
        seen.add(value)
        out.append(value)
    return out


print("Đang nạp dữ liệu...")

df_corpus = pd.read_csv(FILE_CORPUS)
df_corpus = df_corpus.drop_duplicates(subset=[COL_CORPUS_URL])
corpus_dict = df_corpus.set_index(COL_CORPUS_URL)[COL_CORPUS_TEXT].to_dict()

df_refined = pd.read_csv(FILE_REFINED)
df_refined["refined_dict"] = df_refined[COL_REFINED_JSON].apply(lambda x: json.loads(x) if pd.notnull(x) else {})

df_top3 = pd.read_csv(FILE_TOP3_URLS)
df_top3[COL_TOP3_TEXT] = df_top3[COL_TOP3_TEXT].apply(lambda x: safe_json_loads(x, []))
df_top3[COL_TOP3_IMAGE] = df_top3[COL_TOP3_IMAGE].apply(lambda x: safe_json_loads(x, []))
df_top3["top_urls_dedup"] = df_top3.apply(
    lambda row: dedupe_keep_order(list(row[COL_TOP3_TEXT]) + list(row[COL_TOP3_IMAGE])[:1]),
    axis=1,
)

df_merged = pd.merge(df_top3, df_refined, on=COL_CLAIM_ID, how="inner", suffixes=("", "_refined"))

print(f"Số claim: {len(df_merged)}")
print(f"Số URL evidence trung bình mỗi claim: {df_merged['top_urls_dedup'].apply(len).mean():.2f}")
df_merged[[COL_CLAIM_ID, "claim", "label", COL_TOP3_TEXT, COL_TOP3_IMAGE, "top_urls_dedup"]].head()

Đang nạp dữ liệu...
Số claim: 1293
Số URL evidence trung bình mỗi claim: 3.15


,id,claim,label,top3_text_urls,top3_image_urls,top_urls_dedup
0,4,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đảo này ước tính là 100 tỷ đồng.,nei,"[https://tuoitre.vn/3-nhom-nguoi-nuoc-ngoai-lap-cu-diem-o-bac-ninh-hoat-dong-lua-dao-cong-nghe-cao-20260319193417625.htm, https://bocongan.gov.vn/bai-viet/bac-ninh-lien-tiep-ph...","[https://tuoitre.vn/tong-lanh-su-ha-lan-tham-bao-tuoi-tre-mong-ket-noi-nhieu-hon-voi-gioi-tre-viet-nam-20260319143253948.htm, https://vnexpress.net/dien-vien-quang-minh-toi-tan...","[https://tuoitre.vn/3-nhom-nguoi-nuoc-ngoai-lap-cu-diem-o-bac-ninh-hoat-dong-lua-dao-cong-nghe-cao-20260319193417625.htm, https://bocongan.gov.vn/bai-viet/bac-ninh-lien-tiep-ph..."
1,6,Đường dây lừa đảo này chỉ nhắm mục tiêu vào người bị hại tại tỉnh Thái Bình.,nei,"[https://www.facebook.com/mps.gov/posts/pfbid0RXyaU2k459DeBXLcx6V6dSyVn6aYor9DCZa4cTeHoMGgQFDJM9ycwt6vSqBj5NNql, https://www.facebook.com/mps.gov/posts/pfbid02foZytdE1pqfVkCmNa...","[https://vnexpress.net/oppo-find-n6-smartphone-an-nep-gap-ho-tro-but-ai-5052845.html, https://vnexpress.net/oppo-find-n6-smartphone-an-nep-gap-ho-tro-but-ai-5052845.html, https...","[https://www.facebook.com/mps.gov/posts/pfbid0RXyaU2k459DeBXLcx6V6dSyVn6aYor9DCZa4cTeHoMGgQFDJM9ycwt6vSqBj5NNql, https://www.facebook.com/mps.gov/posts/pfbid02foZytdE1pqfVkCmNa..."
2,5,Các đối tượng lừa đảo đã bị bắt giữ vào ngày 15 tháng 5 năm 2024.,nei,"[https://www.facebook.com/mps.gov/posts/pfbid0247VUq4DxaXynPzTbZpUqhvL2c5C77LKHPZswsnMxfHT51M1WTwTRB4hkrUE9FJgTl, https://www.facebook.com/mps.gov/posts/pfbid0sGma2JDRxMm6J7ooL...","[https://bocongan.gov.vn/bai-viet/no-luc-ngan-chan-khai-thac-thuy-san-bat-hop-phap-tai-vung-bien-cuc-nam-to-quoc-1773562721, https://www.facebook.com/thongtinchinhphu/posts/pfb...","[https://www.facebook.com/mps.gov/posts/pfbid0247VUq4DxaXynPzTbZpUqhvL2c5C77LKHPZswsnMxfHT51M1WTwTRB4hkrUE9FJgTl, https://www.facebook.com/mps.gov/posts/pfbid0sGma2JDRxMm6J7ooL..."
3,1,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng trong đường dây lừa đảo hỗ trợ vay vốn online.,supported,"[https://www.facebook.com/mps.gov/posts/pfbid02foZytdE1pqfVkCmNa7gY5z7Vj1zSDX3o3rnniCxgMJxHCm5UJsYKm1h8zRPbfUuXl, https://www.facebook.com/mps.gov/posts/pfbid02foZytdE1pqfVkCmN...","[https://vnexpress.net/xe-may-dien-trung-quoc-dung-cam-bien-lidar-nhu-oto-5052339.html, https://vnexpress.net/vinmec-ocean-park-2-van-hanh-may-mri-3-0-5052072.html, https://vne...","[https://www.facebook.com/mps.gov/posts/pfbid02foZytdE1pqfVkCmNa7gY5z7Vj1zSDX3o3rnniCxgMJxHCm5UJsYKm1h8zRPbfUuXl, https://www.facebook.com/mps.gov/posts/pfbid0RXyaU2k459DeBXLcx..."
4,2,Các đối tượng cầm đầu đường dây lừa đảo đã thuê nhà tại Hà Nội và TP. Hồ Chí Minh để thực hiện hành vi phạm tội.,supported,"[https://tuoitre.vn/bo-cua-mr-pips-nho-nguoi-chay-benh-an-tam-than-cho-con-va-bi-lua-tien-ti-20260319113914643.htm, https://www.facebook.com/mps.gov/posts/pfbid02Poq4xxkZKwqVWQ...","[https://www.facebook.com/mps.gov/posts/pfbid024cYNexz1XZ1CUL47KVndYMqfA7Fm3MLbkzLpWWQ36Fs1gdvBkSocuM2fKpujpPyKl, https://vnexpress.net/cach-cac-nuoc-quan-ly-xe-ban-tai-trong-t...","[https://tuoitre.vn/bo-cua-mr-pips-nho-nguoi-chay-benh-an-tam-than-cho-con-va-bi-lua-tien-ti-20260319113914643.htm, https://www.facebook.com/mps.gov/posts/pfbid02Poq4xxkZKwqVWQ..."


## 3. Smoke Test Dữ Liệu

Cell này không gọi API. Nó chỉ kiểm tra vài claim đầu tiên sẽ dùng bao nhiêu evidence URL.

In [5]:

smoke_frame = df_merged.head(SMOKE_ROWS).copy()
smoke_rows = []
for _, row in smoke_frame.iterrows():
    urls = row["top_urls_dedup"]
    found = [url for url in urls if url in corpus_dict]
    missing = [url for url in urls if url not in corpus_dict]
    smoke_rows.append({
        "id": row[COL_CLAIM_ID],
        "claim": row["claim"],
        "label": row["label"],
        "urls_after_dedup": len(urls),
        "urls_found_in_corpus": len(found),
        "urls_missing_in_corpus": len(missing),
        "urls_used": found,
    })

df_smoke_check = pd.DataFrame(smoke_rows)
df_smoke_check


,id,claim,label,urls_after_dedup,urls_found_in_corpus,urls_missing_in_corpus,urls_used
0,4,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đảo này ước tính là 100 tỷ đồng.,nei,4,4,0,"[https://tuoitre.vn/3-nhom-nguoi-nuoc-ngoai-lap-cu-diem-o-bac-ninh-hoat-dong-lua-dao-cong-nghe-cao-20260319193417625.htm, https://bocongan.gov.vn/bai-viet/bac-ninh-lien-tiep-ph..."
1,6,Đường dây lừa đảo này chỉ nhắm mục tiêu vào người bị hại tại tỉnh Thái Bình.,nei,4,4,0,"[https://www.facebook.com/mps.gov/posts/pfbid0RXyaU2k459DeBXLcx6V6dSyVn6aYor9DCZa4cTeHoMGgQFDJM9ycwt6vSqBj5NNql, https://www.facebook.com/mps.gov/posts/pfbid02foZytdE1pqfVkCmNa..."
2,5,Các đối tượng lừa đảo đã bị bắt giữ vào ngày 15 tháng 5 năm 2024.,nei,4,4,0,"[https://www.facebook.com/mps.gov/posts/pfbid0247VUq4DxaXynPzTbZpUqhvL2c5C77LKHPZswsnMxfHT51M1WTwTRB4hkrUE9FJgTl, https://www.facebook.com/mps.gov/posts/pfbid0sGma2JDRxMm6J7ooL..."
3,1,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng trong đường dây lừa đảo hỗ trợ vay vốn online.,supported,3,3,0,"[https://www.facebook.com/mps.gov/posts/pfbid02foZytdE1pqfVkCmNa7gY5z7Vj1zSDX3o3rnniCxgMJxHCm5UJsYKm1h8zRPbfUuXl, https://www.facebook.com/mps.gov/posts/pfbid0RXyaU2k459DeBXLcx..."
4,2,Các đối tượng cầm đầu đường dây lừa đảo đã thuê nhà tại Hà Nội và TP. Hồ Chí Minh để thực hiện hành vi phạm tội.,supported,4,4,0,"[https://tuoitre.vn/bo-cua-mr-pips-nho-nguoi-chay-benh-an-tam-than-cho-con-va-bi-lua-tien-ti-20260319113914643.htm, https://www.facebook.com/mps.gov/posts/pfbid02Poq4xxkZKwqVWQ..."


## 3. Schema Và Prompt

In [6]:
Relation = Literal["SUPPORT", "REFUTE", "PARTIAL_SUPPORT", "UNRELATED"]
Verdict = Literal["SUPPORTED", "REFUTED", "NEI"]


class EvidenceJudge(BaseModel):
    model_config = ConfigDict(extra="forbid")

    thought_process: str
    relation: Relation
    extracted_facts: str


class FinalThoughtProcess(BaseModel):
    model_config = ConfigDict(extra="forbid")

    synthesis: str
    target_check: str
    logical_deduction: str


class FinalVerdict(BaseModel):
    model_config = ConfigDict(extra="forbid")

    thought_process: FinalThoughtProcess
    verdict: Verdict
    explanation: str


EVIDENCE_SCHEMA = EvidenceJudge.model_json_schema()
FINAL_SCHEMA = FinalVerdict.model_json_schema()


def extract_json(text):
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE | re.DOTALL).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def validate_model(model_cls, raw_text):
    data = extract_json(raw_text)
    return model_cls.model_validate(data).model_dump()

In [7]:
def format_atoms(refined_claim):
    atoms = []
    for atom in refined_claim.get("claim_atoms", []) or []:
        if isinstance(atom, dict):
            text = atom.get("text", "")
            priority = atom.get("priority", "")
            atoms.append(f"+ {text} (Ưu tiên: {priority})" if priority else f"+ {text}")
        elif atom:
            atoms.append(f"+ {atom}")
    return "\n".join(atoms) if atoms else "Không có mệnh đề cụ thể."


def format_visuals(refined_claim):
    visuals = []
    for item in refined_claim.get("visual_observations", []) or []:
        if isinstance(item, dict):
            visuals.append(f"+ {item.get('text', '')}")
        elif item:
            visuals.append(f"+ {item}")
    return "\n".join(visuals) if visuals else "Không có thông tin thị giác."


def format_targets(refined_claim):
    targets = [f"- {t}" for t in refined_claim.get("verification_targets", []) or []]
    return "\n".join(targets) if targets else "Không có mục tiêu cụ thể."


MAP_SYSTEM_PROMPT = """You are a strict fact-checking evidence judge.
Task: Compare a structurally analyzed Claim with ONE piece of Evidence.

Rules:
- Use only information from this Evidence. Do not use external knowledge.
- Treat title, URL, and retrieval rank as metadata, not facts by themselves.
- SUPPORT means the evidence fully supports the relevant claim atom.
- REFUTE means the evidence directly contradicts the relevant claim atom.
- PARTIAL_SUPPORT means the evidence is relevant but incomplete.
- UNRELATED means the evidence does not address the claim.
- Return JSON only, following the schema exactly.
"""

FINAL_SYSTEM_PROMPT = """You are a strict final fact-checking judge.
Use only the extracted evidence facts from the previous step.

Verdict rules:
- SUPPORTED: enough evidence supports all central verification targets.
- REFUTED: at least one central target is directly contradicted by evidence.
- NEI: evidence is missing, unrelated, indirect, or insufficient.
- A direct contradiction has priority over partial support.
- The explanation must be in Vietnamese, under 50 words, natural for an end user.
- Do not mention internal process terms such as source number, map, reduce, retrieval, JSON, or model.
- Return JSON only, following the schema exactly.
"""

## 4. Hàm Gọi OpenRouter

In [8]:
async def call_openrouter_schema(messages, schema_name, schema, max_tokens=1400, retry_note=""):
    if retry_note:
        messages = [
            *messages,
            {"role": "user", "content": f"Output trước đó lỗi validation: {retry_note}. Hãy trả lại JSON hợp lệ, không markdown."},
        ]
    response = await client.chat.completions.create(
        model=CURRENT_MODEL_ID,
        messages=messages,
        temperature=0.0,
        max_tokens=max_tokens,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": schema_name,
                "strict": True,
                "schema": schema,
            },
        },
    )
    return response.choices[0].message.content or ""


async def evaluate_single_evidence(refined_claim, evidence_text, evidence_index, retry_limit=3):
    norm_claim = refined_claim.get("normalized_claim", "")
    user_prompt = f"""[CLAIM STRUCTURE TO VERIFY]
- Claim: {norm_claim}
- Claim Atoms to consider:
{format_atoms(refined_claim)}
- Visual observations from the attached image:
{format_visuals(refined_claim)}

[EVIDENCE {evidence_index}]
{evidence_text}
"""

    messages = [
        {"role": "system", "content": MAP_SYSTEM_PROMPT},
        {"role": "user", "content": [{"type": "text", "text": user_prompt}]},
    ]

    retry_note = ""
    last_error = ""
    for attempt in range(1, retry_limit + 1):
        try:
            raw = await call_openrouter_schema(messages, "single_evidence_judge", EVIDENCE_SCHEMA, max_tokens=1200, retry_note=retry_note)
            return validate_model(EvidenceJudge, raw)
        except Exception as e:
            last_error = str(e)
            retry_note = last_error[:1000]
            await asyncio.sleep(min(2 * attempt, 8))

    print(f"Lỗi khi xử lý Evidence {evidence_index}: {last_error}")
    return None


async def get_final_verdict(refined_claim, map_results, retry_limit=3):
    norm_claim = refined_claim.get("normalized_claim", "")

    compiled_facts = ""
    for idx, res in enumerate(map_results):
        if res and res.get("relation") != "UNRELATED":
            compiled_facts += f"\n- Nguồn {idx + 1} ({res.get('relation')}): {res.get('extracted_facts')}"
    if not compiled_facts.strip():
        compiled_facts = "\nNo relevant information regarding the Claim from the provided evidence."

    user_prompt = f"""[CLAIM]
{norm_claim}

[VERIFICATION TARGETS]
{format_targets(refined_claim)}

[EXTRACTED INFORMATION PIECES FROM EVIDENCE]
{compiled_facts}
"""

    messages = [
        {"role": "system", "content": FINAL_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    retry_note = ""
    last_error = ""
    for attempt in range(1, retry_limit + 1):
        try:
            raw = await call_openrouter_schema(messages, "final_fact_check_verdict", FINAL_SCHEMA, max_tokens=1400, retry_note=retry_note)
            return validate_model(FinalVerdict, raw), ""
        except Exception as e:
            last_error = str(e)
            retry_note = last_error[:1000]
            await asyncio.sleep(min(2 * attempt, 8))

    print(f"Lỗi phán quyết cuối: {last_error}")
    return None, last_error

## 5. Chạy Judge

In [9]:
RUN_JUDGE = True 
MAX_CLAIMS_TO_PROCESS = SMOKE_ROWS if SMOKE_TEST else None
MAX_CONCURRENT_REQUESTS = 2 if SMOKE_TEST else 8
SAVE_EVERY = 1 if SMOKE_TEST else 10
MAX_EVIDENCE_CHARS = 4500
RETRY_LIMIT = 3


def load_successful_results(output_file):
    if not output_file.exists():
        return [], set()
    existing = pd.read_csv(output_file)
    if "final_error" in existing.columns:
        ok = existing["final_error"].fillna("").astype(str).str.strip() == ""
        existing = existing[ok].copy()
    completed_ids = set(existing["claim_id"].astype(str)) if "claim_id" in existing.columns else set()
    return existing.to_dict("records"), completed_ids


def write_results(rows, output_file):
    pd.DataFrame(rows).to_csv(output_file, index=False, encoding="utf-8-sig")


async def process_one_claim(row, semaphore):
    async with semaphore:
        claim_id = row[COL_CLAIM_ID]
        refined_claim = row["refined_dict"]
        top_urls = row["top_urls_dedup"]

        evidences_text = []
        urls_used = []
        for url in top_urls:
            if url in corpus_dict:
                text = str(corpus_dict[url])
                if len(text) > MAX_EVIDENCE_CHARS:
                    text = text[:MAX_EVIDENCE_CHARS].rsplit(" ", 1)[0] + "..."
                evidences_text.append(text)
                urls_used.append(url)
            else:
                print(f"[Cảnh báo] Không tìm thấy URL trong corpus: {url}")

        map_results = []
        for idx, ev_text in enumerate(evidences_text):
            result = await evaluate_single_evidence(refined_claim, ev_text, idx + 1, retry_limit=RETRY_LIMIT)
            map_results.append(result)

        final_verdict, final_error = await get_final_verdict(refined_claim, map_results, retry_limit=RETRY_LIMIT)
        if final_verdict is None:
            final_verdict = {"verdict": "NEI", "explanation": "", "thought_process": {}}

        return {
            "claim_id": claim_id,
            "normalized_claim": refined_claim.get("normalized_claim"),
            "verdict": final_verdict.get("verdict"),
            "explanation": final_verdict.get("explanation"),
            "thought_process": json.dumps(final_verdict.get("thought_process", {}), ensure_ascii=False),
            "top3_urls_used": json.dumps(urls_used, ensure_ascii=False),
            "map_results": json.dumps(map_results, ensure_ascii=False),
            "gold_label": row.get("label", ""),
            "final_error": final_error,
        }


async def run_judge():
    rows, completed_ids = load_successful_results(OUTPUT_FILE)
    frame = df_merged.copy()
    if MAX_CLAIMS_TO_PROCESS is not None:
        frame = frame.head(MAX_CLAIMS_TO_PROCESS).copy()
    todo = frame[~frame[COL_CLAIM_ID].astype(str).isin(completed_ids)].copy()

    print(f"Smoke test: {SMOKE_TEST}")
    print(f"Giới hạn claim: {MAX_CLAIMS_TO_PROCESS}")
    print(f"Đã thành công trước đó: {len(completed_ids)}")
    print(f"Còn cần chạy: {len(todo)}")

    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    tasks = [asyncio.create_task(process_one_claim(row, semaphore)) for _, row in todo.iterrows()]

    completed_since_save = 0
    for task in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="judge"):
        rows.append(await task)
        completed_since_save += 1
        if completed_since_save >= SAVE_EVERY:
            write_results(rows, OUTPUT_FILE)
            completed_since_save = 0

    write_results(rows, OUTPUT_FILE)
    print(f"Đã lưu kết quả tại: {OUTPUT_FILE}")
    return pd.DataFrame(rows)


if RUN_JUDGE:
    df_results = await run_judge()
else:
    print("RUN_JUDGE = False, bỏ qua gọi API.")

Smoke test: False
Giới hạn claim: None
Đã thành công trước đó: 0
Còn cần chạy: 1293


judge:   0%|          | 0/1293 [00:00<?, ?it/s]

Đã lưu kết quả tại: d:\FactCheckPipeline\judge\judge_outputs_openrouter_deepseek_v4_flash\factchecking_final_results_deepseek-v4-flash__gemini-2.5-flash__semantic__clip_finetuned__reranker_1__full.csv


## 6. Nạp Kết Quả

In [10]:
if OUTPUT_FILE.exists():
    df_results = pd.read_csv(OUTPUT_FILE)
    print(f"Đã nạp {len(df_results)} dòng từ {OUTPUT_FILE}")
else:
    df_results = pd.DataFrame()
    print(f"Chưa có output: {OUTPUT_FILE}")

df_results.head()

Đã nạp 1293 dòng từ d:\FactCheckPipeline\judge\judge_outputs_openrouter_deepseek_v4_flash\factchecking_final_results_deepseek-v4-flash__gemini-2.5-flash__semantic__clip_finetuned__reranker_1__full.csv


,claim_id,normalized_claim,verdict,explanation,thought_process,top3_urls_used,map_results,gold_label,final_error
0,3,Thượng úy Nguyễn Đức Phước là điều tra viên thụ lý chính của vụ án lừa đảo này.,REFUTED,"Thông tin từ nguồn đối chiếu cho thấy điều tra viên thụ lý chính là Thượng úy Nguyễn Văn Thái, còn Nguyễn Đức Phước là Đại úy, Phó Đội trưởng. Khớp với tên vụ án lừa đảo nhưng ...","{""synthesis"": ""Người dùng khẳng định Thượng úy Nguyễn Đức Phước là điều tra viên thụ lý chính vụ án lừa đảo. Từ dữ liệu trích xuất, nguồn 1 trực tiếp mâu thuẫn: 'Thượng úy Nguy...","[""https://www.facebook.com/mps.gov/posts/pfbid02foZytdE1pqfVkCmNa7gY5z7Vj1zSDX3o3rnniCxgMJxHCm5UJsYKm1h8zRPbfUuXl"", ""https://bocongan.gov.vn/bai-viet/triet-pha-bang-nhom-lua-da...","[{""thought_process"": ""I need to evaluate the evidence against each claim atom separately. First, 'Thượng úy Nguyễn Đức Phước là điều tra viên thụ lý chính.' The evidence says: ...",refuted,NaN
1,2,Các đối tượng cầm đầu đường dây lừa đảo đã thuê nhà ở Hà Nội và TP. Hồ Chí Minh để thực hiện hành vi phạm tội.,NEI,"Có thông tin thuê văn phòng tại Hà Nội để tổ chức đánh bạc, nhưng không đề cập đến TP. Hồ Chí Minh hay hành vi lừa đảo. Do đó, chưa đủ bằng chứng để xác nhận hoàn toàn.","{""synthesis"": ""Nguồn 3 xác nhận thuê văn phòng tại Hà Nội để hoạt động tội phạm (tổ chức đánh bạc), nhưng không phải lừa đảo. Không có thông tin về TP. Hồ Chí Minh. Mục đích th...","[""https://tuoitre.vn/bo-cua-mr-pips-nho-nguoi-chay-benh-an-tam-than-cho-con-va-bi-lua-tien-ti-20260319113914643.htm"", ""https://www.facebook.com/mps.gov/posts/pfbid02Poq4xxkZKwq...","[{""thought_process"": ""The evidence describes Mr Pips (Phó Đức Nam) as the ringleader of a fraud scheme. It states that he 'bắt đầu thuê văn phòng, tuyển nhân viên để lên kế hoạ...",supported,NaN
2,4,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đảo này ước tính là 100 tỷ đồng.,REFUTED,"Không có bằng chứng nào cho thấy số tiền bị chiếm đoạt là 100 tỷ đồng. Các nguồn tin đưa ra con số khác, như 21 tỷ hoặc 160 tỷ đồng.","{""synthesis"": ""Các nguồn đều không khớp với số tiền 100 tỷ đồng. Nguồn 2 nói 21 tỷ đồng, nguồn 3 nói 160 tỷ đồng. Không có bằng chứng nào hỗ trợ con số 100 tỷ đồng."", ""target_c...","[""https://tuoitre.vn/3-nhom-nguoi-nuoc-ngoai-lap-cu-diem-o-bac-ninh-hoat-dong-lua-dao-cong-nghe-cao-20260319193417625.htm"", ""https://bocongan.gov.vn/bai-viet/bac-ninh-lien-tiep...","[{""thought_process"": ""The claim states that the total amount defrauded by this scam ring is estimated at 100 billion VND. The evidence describes three separate scam rings. For ...",nei,NaN
3,1,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng liên quan đến đường dây lừa đảo hỗ trợ vay vốn trực tuyến.,SUPPORTED,"Công an tỉnh Thái Bình đã khởi tố 10 đối tượng trong đường dây lừa đảo hỗ trợ vay vốn trực tuyến, thông tin hoàn toàn chính xác.","{""synthesis"": ""Các mảnh bằng chứng từ Nguồn 1 khẳng định: Công an tỉnh Thái Bình khởi tố 10 đối tượng, các đối tượng thuộc đường dây lừa đảo hỗ trợ vay vốn trực tuyến. Cả ba mụ...","[""https://www.facebook.com/mps.gov/posts/pfbid02foZytdE1pqfVkCmNa7gY5z7Vj1zSDX3o3rnniCxgMJxHCm5UJsYKm1h8zRPbfUuXl"", ""https://www.facebook.com/mps.gov/posts/pfbid0RXyaU2k459DeBX...","[{""thought_process"": ""The evidence explicitly states: 'Cơ quan Cảnh sát điều tra Công an tỉnh Thái Bình đã ra quyết định khởi tố vụ án, khởi tố bị can đối với 10 đối tượng liên...",supported,NaN
4,9,"Chỉ có 5 đối tượng, trong đó có Bùi Quốc Ý, bị Phòng Cảnh sát hình sự Công an tỉnh Thanh Hóa tạm giữ hình sự liên quan đến vụ án này.",REFUTED,"Tuyên bố cho rằng chỉ có 5 đối tượng bị tạm giữ, nhưng bằng chứng cho thấy Phòng Cảnh sát hình sự Thanh Hóa đã tạm giữ 8 đối tượng, trong đó có Bùi Quốc Ý, nên thông tin này kh...","{""synthesis"": ""Nguồn 1 xác nhận danh tính Bùi Quốc Ý và đơn vị Phòng Cảnh sát hình sự Công an tỉnh Thanh Hóa, nhưng số lượng đối tượng là 8, mâu thuẫn với con số 5 trong tuyên ...","[""https://www.facebook.com/mps.gov/posts/pfbid02LzLmULGqiVgr

## 7. Metrics

In [11]:
LABELS = ["supported", "refuted", "nei"]
VERDICT_TO_LABEL = {"SUPPORTED": "supported", "REFUTED": "refuted", "NEI": "nei"}


def normalize_gold_label(value):
    text = str(value or "").strip().lower()
    return text if text in LABELS else ""


def normalize_pred_label(value):
    text = str(value or "").strip().upper()
    return VERDICT_TO_LABEL.get(text, "")


def classification_report_frame(y_true, y_pred, labels):
    rows = []
    total = len(y_true)
    for label in labels:
        tp = int(((y_true == label) & (y_pred == label)).sum())
        fp = int(((y_true != label) & (y_pred == label)).sum())
        fn = int(((y_true == label) & (y_pred != label)).sum())
        support = int((y_true == label).sum())
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"label": label, "precision": precision, "recall": recall, "f1": f1, "support": support, "tp": tp, "fp": fp, "fn": fn})

    report = pd.DataFrame(rows)
    accuracy = float((y_true == y_pred).mean()) if total else 0.0
    macro = report[["precision", "recall", "f1"]].mean(numeric_only=True)
    weights = report["support"] / report["support"].sum() if report["support"].sum() else 0
    weighted = (report[["precision", "recall", "f1"]].multiply(weights, axis=0)).sum()

    summary = pd.DataFrame([
        {"label": "accuracy", "precision": math.nan, "recall": math.nan, "f1": accuracy, "support": total, "tp": math.nan, "fp": math.nan, "fn": math.nan},
        {"label": "macro_avg", "precision": macro["precision"], "recall": macro["recall"], "f1": macro["f1"], "support": total, "tp": math.nan, "fp": math.nan, "fn": math.nan},
        {"label": "weighted_avg", "precision": weighted["precision"], "recall": weighted["recall"], "f1": weighted["f1"], "support": total, "tp": math.nan, "fp": math.nan, "fn": math.nan},
    ])
    return pd.concat([report, summary], ignore_index=True)


if df_results.empty:
    df_eval = pd.DataFrame()
    df_report = pd.DataFrame()
    df_confusion = pd.DataFrame()
    df_errors = pd.DataFrame()
    print("Chưa có kết quả judge để đánh giá.")
else:
    df_gold = pd.read_csv(FILE_GOLD)[["id", "claim", "label"]].copy()
    df_gold["gold_label"] = df_gold["label"].apply(normalize_gold_label)

    df_pred = df_results.copy()
    if "final_error" in df_pred.columns:
        df_pred = df_pred[df_pred["final_error"].fillna("").astype(str).str.strip() == ""].copy()

    required_pred_cols = {"claim_id", "verdict"}
    missing_pred_cols = required_pred_cols - set(df_pred.columns)
    if missing_pred_cols:
        raise ValueError(f"Output judge thiếu cột bắt buộc: {sorted(missing_pred_cols)}")

    # Output judge có thể đã lưu gold_label. Bỏ cột này để tránh pandas tạo gold_label_gold/gold_label_pred sau merge.
    df_pred = df_pred.drop(columns=["gold_label"], errors="ignore")
    df_pred["pred_label"] = df_pred["verdict"].apply(normalize_pred_label)
    df_pred = df_pred.drop_duplicates(subset=["claim_id"], keep="last")

    df_eval = df_gold.merge(df_pred, left_on="id", right_on="claim_id", how="inner")
    df_eval = df_eval[(df_eval["gold_label"] != "") & (df_eval["pred_label"] != "")].copy()
    df_errors = df_eval[df_eval["gold_label"] != df_eval["pred_label"]].copy()

    df_report = classification_report_frame(df_eval["gold_label"], df_eval["pred_label"], LABELS)
    df_confusion = pd.crosstab(df_eval["gold_label"], df_eval["pred_label"], rownames=["gold"], colnames=["pred"]).reindex(index=LABELS, columns=LABELS, fill_value=0)

    print(f"Số dòng được đánh giá: {len(df_eval)}")

df_report

Số dòng được đánh giá: 1293


,label,precision,recall,f1,support,tp,fp,fn
0,supported,0.953168,0.800926,0.870440,432,346.0,17.0,86.0
1,refuted,0.686679,0.849188,0.759336,431,366.0,167.0,65.0
2,nei,0.740554,0.683721,0.711004,430,294.0,103.0,136.0
3,accuracy,NaN,NaN,0.778036,1293,NaN,NaN,NaN
4,macro_avg,0.793467,0.777945,0.780260,1293,NaN,NaN,NaN
5,weighted_avg,0.793632,0.778036,0.780383,1293,NaN,NaN,NaN


In [12]:
df_confusion

pred,supported,refuted,nei
gold,,,
supported,346,39,47
refuted,9,366,56
nei,8,128,294


## 8. Lỗi Và Lưu File

In [13]:
if df_eval.empty:
    print("Chưa có dòng đánh giá.")
else:
    print(f"Số lỗi: {len(df_errors)} / {len(df_eval)}")
    display(
        df_errors.groupby(["gold_label", "pred_label"])
        .size()
        .rename("so_luong")
        .reset_index()
        .sort_values("so_luong", ascending=False)
    )

    cols = ["id", "claim", "gold_label", "pred_label", "verdict", "explanation", "top3_urls_used", "map_results"]
    cols = [c for c in cols if c in df_errors.columns]
    display(df_errors[cols].head(20))

Số lỗi: 287 / 1293


,gold_label,pred_label,so_luong
0,nei,refuted,128
2,refuted,nei,56
4,supported,nei,47
5,supported,refuted,39
3,refuted,supported,9
1,nei,supported,8


,id,claim,gold_label,pred_label,verdict,explanation,top3_urls_used,map_results
1,2,Các đối tượng cầm đầu đường dây lừa đảo đã thuê nhà tại Hà Nội và TP. Hồ Chí Minh để thực hiện hành vi phạm tội.,supported,nei,NEI,"Có thông tin thuê văn phòng tại Hà Nội để tổ chức đánh bạc, nhưng không đề cập đến TP. Hồ Chí Minh hay hành vi lừa đảo. Do đó, chưa đủ bằng chứng để xác nhận hoàn toàn.","[""https://tuoitre.vn/bo-cua-mr-pips-nho-nguoi-chay-benh-an-tam-than-cho-con-va-bi-lua-tien-ti-20260319113914643.htm"", ""https://www.facebook.com/mps.gov/posts/pfbid02Poq4xxkZKwq...","[{""thought_process"": ""The evidence describes Mr Pips (Phó Đức Nam) as the ringleader of a fraud scheme. It states that he 'bắt đầu thuê văn phòng, tuyển nhân viên để lên kế hoạ..."
3,4,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đảo này ước tính là 100 tỷ đồng.,nei,refuted,REFUTED,"Không có bằng chứng nào cho thấy số tiền bị chiếm đoạt là 100 tỷ đồng. Các nguồn tin đưa ra con số khác, như 21 tỷ hoặc 160 tỷ đồng.","[""https://tuoitre.vn/3-nhom-nguoi-nuoc-ngoai-lap-cu-diem-o-bac-ninh-hoat-dong-lua-dao-cong-nghe-cao-20260319193417625.htm"", ""https://bocongan.gov.vn/bai-viet/bac-ninh-lien-tiep...","[{""thought_process"": ""The claim states that the total amount defrauded by this scam ring is estimated at 100 billion VND. The evidence describes three separate scam rings. For ..."
4,5,Các đối tượng lừa đảo đã bị bắt giữ vào ngày 15 tháng 5 năm 2024.,nei,refuted,REFUTED,"Có bằng chứng bắt giữ đối tượng lừa đảo, nhưng không xác nhận thời gian là ngày 15/5/2024. Không tìm thấy thông tin này trong chứng cứ.","[""https://www.facebook.com/mps.gov/posts/pfbid0247VUq4DxaXynPzTbZpUqhvL2c5C77LKHPZswsnMxfHT51M1WTwTRB4hkrUE9FJgTl"", ""https://www.facebook.com/mps.gov/posts/pfbid0sGma2JDRxMm6J7...","[{""thought_process"": ""The claim consists of two atoms: (1) 'Các đối tượng lừa đảo đã bị bắt giữ' (The fraud subjects were arrested) and (2) 'Việc bắt giữ diễn ra vào ngày 15 th..."
5,6,Đường dây lừa đảo này chỉ nhắm mục tiêu vào người bị hại tại tỉnh Thái Bình.,nei,refuted,REFUTED,"Bằng chứng cho thấy đường dây lừa đảo này có hơn 200 bị hại trên cả nước, không chỉ riêng tỉnh Thái Bình. Các đối tượng cầm đầu ở Hà Nội và TP.HCM, hoạt động qua mạng xã hội, n...","[""https://www.facebook.com/mps.gov/posts/pfbid0RXyaU2k459DeBXLcx6V6dSyVn6aYor9DCZa4cTeHoMGgQFDJM9ycwt6vSqBj5NNql"", ""https://www.facebook.com/mps.gov/posts/pfbid02foZytdE1pqfVkC...","[{""thought_process"": ""The evidence states that the scam ring involves victims from many provinces (\""hơn 200 bị hại trên cả nước\""). This directly contradicts the claim's state..."
17,18,"Đêm giao thừa Tết Bính Ngọ 2026, các sản phụ chờ sinh chủ yếu tập trung tại Bệnh viện Phụ sản Trung ương ở Hà Nội.",nei,refuted,REFUTED,"Bằng chứng cho thấy đêm giao thừa Tết Bính Ngọ 2026, có khoảng 30 sản phụ chờ sinh tại Bệnh viện Từ Dũ (TPHCM), trái ngược với thông tin trong tuyên bố.","[""https://www.facebook.com/thongtinchinhphu/posts/pfbid0WXpWRuouxyNDV8V2gRxmaqdjDLJewrJUEWFpsnGD9yGYL2CBpA18vriuiyXXAycVl"", ""https://www.facebook.com/thongtinchinhphu/posts/pfb...","[{""thought_process"": ""I need to check two claim atoms against the provided evidence. The evidence is about hospitals during Tet Binh Ngo 2026.\n\nClaim atom 1: 'Đêm giao thừa T..."
18,19,Phó Thủ tướng Hồ Quốc Dũng yêu cầu các lực lượng phải hành động nhanh nhất để cứu dân trong vùng lũ lụt.,supported,refuted,REFUTED,"Có nguồn chính thức cho thấy yêu cầu này do Phó Thủ tướng Nguyễn Hòa Bình đưa ra, không phải ông Hồ Quốc Dũng.","[""https://www.facebook.com/thongtinchinhphu/posts/pfbid0cd1Ez6yxUKjjityZyARnKgQchp42GXnbHePXQzuCgXdazL6sMBN9Jeoopp56rN24l"", ""https://www.facebook.com/thongtinchinhphu/posts/pfb...","[{""thought_process"": ""The evidence directly quotes Phó Thủ tướng Hồ Quốc Dũng stating 'Tất cả các lực lượng phải hành động nhanh nhất, mạnh nhất, quyết liệt nhất để cứu dân.' T..."
21,22,Phó Thủ tướng Hồ Quốc Dũng yêu cầu thả hàng cứu trợ từ trực thăng với tổng khối lượng 5 tấn

In [14]:
if not df_report.empty:
    df_report.to_csv(METRICS_FILE, index=False, encoding="utf-8-sig")
    df_confusion.to_csv(CONFUSION_FILE, encoding="utf-8-sig")
    df_errors.to_csv(ERRORS_FILE, index=False, encoding="utf-8-sig")
    print(f"Đã lưu metrics: {METRICS_FILE}")
    print(f"Đã lưu confusion matrix: {CONFUSION_FILE}")
    print(f"Đã lưu lỗi: {ERRORS_FILE}")
else:
    print("Chưa có metrics để lưu.")

Đã lưu metrics: d:\FactCheckPipeline\judge\judge_outputs_openrouter_deepseek_v4_flash\factchecking_metrics_deepseek-v4-flash__gemini-2.5-flash__semantic__clip_finetuned__reranker_1__full.csv
Đã lưu confusion matrix: d:\FactCheckPipeline\judge\judge_outputs_openrouter_deepseek_v4_flash\factchecking_confusion_deepseek-v4-flash__gemini-2.5-flash__semantic__clip_finetuned__reranker_1__full.csv
Đã lưu lỗi: d:\FactCheckPipeline\judge\judge_outputs_openrouter_deepseek_v4_flash\factchecking_errors_deepseek-v4-flash__gemini-2.5-flash__semantic__clip_finetuned__reranker_1__full.csv
